# Ropedia Academy — D2 · SLAM reuses Track B's geometry

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/D2.ipynb)

> **Solves PnP (camera pose from 3D↔2D) by reprojection minimization and maps every SLAM step back to its Track B operation — including the move to neural/dense maps.**
>
> 用重投影最小化求解 PnP（从 3D↔2D 求相机位姿），并把每个 SLAM 步骤映射回其 Track B 操作——包括转向神经/稠密地图。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/D2

In [ ]:
import torch

# SLAM = Track B geometry, online. TRACKING each frame = PnP (pose from 3D<->2D).
K = torch.tensor([[600,0,320.],[0,600,240.],[0,0,1.]])
pts3d = torch.randn(40, 3) + torch.tensor([0, 0, 4.])     # known map points
R, t_true = torch.eye(3), torch.tensor([0.3, -0.2, 0.1])
def project(P, t):
    p = P @ R.T + t; uv = (K @ p.T).T; return uv[:, :2] / uv[:, 2:3]
obs = project(pts3d, t_true)                               # their 2D detections

# PnP: recover the camera pose by minimizing reprojection error (gradient descent).
t = torch.zeros(3, requires_grad=True)
opt = torch.optim.Adam([t], lr=0.02)
for _ in range(300):
    opt.zero_grad(); ((project(pts3d, t) - obs) ** 2).mean().backward(); opt.step()
print("PnP recovered translation:", t.detach().round(decimals=2).tolist(), "| true:", t_true.tolist())
print("tracking=PnP, mapping=triangulation, back-end=bundle adjustment (B3)")
print("neural/dense SLAM swaps the sparse point map for a renderable NeRF/Gaussian field")

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/D2
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks